# BERT Family Model Comparison for Sarcasm Detection
This notebook trains multiple transformer models on the same dataset and compares their performance.

## 1. Install and Import Required Libraries

In [ ]:
!pip install transformers datasets scikit-learn pandas torch -q

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## 2. Load Dataset
Replace `sarcasm_dataset.csv` with your dataset path.

In [ ]:
df = pd.read_csv("sarcasm_dataset.csv")

# Expecting columns: text, label
df = df[['text','label']]

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

## 3. Define Models to Compare

In [ ]:
models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "ALBERT": "albert-base-v2",
    "DeBERTa": "microsoft/deberta-base"
}

## 4. Tokenization Function

In [ ]:
def tokenize_function(example, tokenizer):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

## 5. Metrics Function

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## 6. Training Loop for All Models

In [ ]:
results = {}

for name, model_name in models.items():
    
    print(f"\nTraining {name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    tokenized_dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./results_{name}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_dir="./logs",
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()
    
    results[name] = metrics["eval_accuracy"]

print(results)

## 7. Display Comparison Results

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=["Model","Accuracy"])
results_df.sort_values(by="Accuracy", ascending=False)

## 8. Plot Accuracy Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("BERT Family Model Accuracy Comparison")
plt.show()